#### Real-time voice transcription using Transcribe
#### You need to set the AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY through terminal on your aws config
#### The AWS permissio policy for the role:
{
	"Version": "2012-10-17",
	"Statement": [
		{
			"Sid": "VisualEditor0",
			"Effect": "Allow",
			"Action": [
				"transcribe:GetTranscriptionJob",
				"transcribe:StartTranscriptionJob",
				"transcribe:ListTranscriptionJobs",
				"transcribe:DeleteTranscriptionJob",
				"transcribe:CreateVocabulary",
				"transcribe:GetVocabulary",
				"transcribe:StartStreamTranscription",
				"transcribe:StartStreamTranscriptionWebSocket"
			],
			"Resource": "*"
		}
	]
}

In [ ]:
!pip install amazon-transcribe sounddevice

###### This code takes your voice and turns it to transcription in real-time.
###### It records your voice until you say the QUIT_PHRASE.
###### I am not using vocabulary table for this implementation. You can add yours.

In [4]:
import asyncio
import re
import numpy as np
import sounddevice as sd
import re
from IPython.display import clear_output

from amazon_transcribe.client import TranscribeStreamingClient
from amazon_transcribe.handlers import TranscriptResultStreamHandler
from amazon_transcribe.model import TranscriptEvent

In [5]:
REGION = "us-east-1"
LANGUAGE_CODE = "en-US"
SAMPLE_RATE = 48000
CHUNK_MS = 20
FRAMES_PER_CHUNK = int(SAMPLE_RATE * CHUNK_MS / 1000)

VOCAB_NAME = None  # "VocabularyTable" (name only)

In [ ]:
import asyncio
import re
import numpy as np
import sounddevice as sd

from amazon_transcribe.client import TranscribeStreamingClient
from amazon_transcribe.handlers import TranscriptResultStreamHandler
from amazon_transcribe.model import TranscriptEvent

from IPython.display import clear_output

REGION = "us-east-1"
LANGUAGE_CODE = "en-US"

INPUT_DEVICE_INDEX = 17   # You cneed to find the best channel for your mic. Use the "check_mic.ipynb to find the best performing channel."
SAMPLE_RATE = 48000 # This is the sample rate for my system, you will find yours.
CHUNK_MS = 20
FRAMES_PER_CHUNK = int(SAMPLE_RATE * CHUNK_MS / 1000)

VOCAB_NAME = None  # "VocabularyTable" you can add your vocabulary table. It has to be created on AWS side.
QUIT_PHRASE = "end of statement"


def _normalize(text: str) -> str:
    # lowercase, remove extra spaces and punctuation-ish for robust quit matching
    text = text.lower()
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


class AccumulatingTranscriptHandler(TranscriptResultStreamHandler):
    def __init__(self, transcript_result_stream, stop_event: asyncio.Event, quit_phrase: str):
        super().__init__(transcript_result_stream)
        self.stop_event = stop_event
        self.quit_phrase = _normalize(quit_phrase)
        self.final_text = ""      # final text
        self.partial_text = ""

    def _render(self):
        clear_output(wait=True)
        combined = (self.final_text + " " + self.partial_text).strip()
        print(combined, flush=True)

    def _check_quit(self, text: str):
        # If quit phrase appears anywhere in the combined text, stop
        combined_norm = _normalize((self.final_text + " " + text).strip())
        if self.quit_phrase in combined_norm:
            self.stop_event.set()

    async def handle_transcript_event(self, transcript_event: TranscriptEvent):
        for result in transcript_event.transcript.results:
            if not result.alternatives:
                continue

            text = (result.alternatives[0].transcript or "").strip()
            if not text:
                continue

            if result.is_partial:
                self.partial_text = text
                self._render()
                self._check_quit(text)
            else:
                # finalize: append and clear partial
                if self.final_text:
                    self.final_text += " " + text
                else:
                    self.final_text = text
                self.partial_text = ""
                self._render()
                self._check_quit(text)


async def run_realtime_transcribe_accumulate():
    sd.default.device = (INPUT_DEVICE_INDEX, None)

    client = TranscribeStreamingClient(region=REGION)
    stream = await client.start_stream_transcription(
        language_code=LANGUAGE_CODE,
        media_sample_rate_hz=SAMPLE_RATE,
        media_encoding="pcm",
        vocabulary_name=VOCAB_NAME,
    )

    loop = asyncio.get_running_loop()
    q: asyncio.Queue[bytes] = asyncio.Queue()
    stop_event = asyncio.Event()

    def callback(indata, frames, time_info, status):
        pcm16 = (np.clip(indata, -1, 1) * 32767).astype(np.int16).tobytes()
        loop.call_soon_threadsafe(q.put_nowait, pcm16)

    async def write_audio():
        with sd.InputStream(
            device=INPUT_DEVICE_INDEX,
            samplerate=SAMPLE_RATE,
            channels=1,
            dtype="float32",
            blocksize=FRAMES_PER_CHUNK,
            callback=callback,
        ):
            while not stop_event.is_set():
                chunk = await q.get()
                await stream.input_stream.send_audio_event(audio_chunk=chunk)
        await stream.input_stream.end_stream()

    handler = AccumulatingTranscriptHandler(stream.output_stream, stop_event, QUIT_PHRASE)

    audio_task = asyncio.create_task(write_audio())
    events_task = asyncio.create_task(handler.handle_events())

    await stop_event.wait()
    await audio_task

    await asyncio.sleep(0.5)

    if not events_task.done():
        events_task.cancel()
        try:
            await events_task
        except asyncio.CancelledError:
            pass

    clear_output(wait=True)
    print("Final text:\n")
    print(handler.final_text.strip(), flush=True)
    return handler.final_text.strip()


final_text = await run_realtime_transcribe_accumulate()
